# Hands-on Demo 3 — Docker Multi-Stage Builds

This notebook is the runnable version of the hands-on demo comparing a bloated single-stage Docker build against a slim multi-stage build for a small Go HTTP server. Run cells top to bottom to build both images, compare their sizes, inspect their layers, and run the app. All commands assume the current working directory is `hands-on-3` (where this notebook lives).

## Summary

- **Step 1 — Prerequisites**: Verify Docker and `dive` are installed.
- **Step 2 — Build the "Before" (Single-Stage Image)**: Build `hello-go:single` from `Dockerfile-single-stage`.
- **Step 3 — Build the "After" (Multi-Stage Image)**: Build `hello-go:multi` from `Dockerfile-multistage`.
- **Step 4 — Compare Image Sizes**: List both images side by side to see the size difference.
- **Step 5 — Tag the Image**: Add a `hello-go:latest` tag to the multi-stage image.
- **Step 6 — Inspect Image Layers with Dive**: Explore each image's layers with `dive`.
- **Step 7 — Run and Verify the App**: Run the multi-stage image and confirm the HTTP server responds.
- **Teardown**: Stop the running demo container and remove the images built during the demo.

---

## Step 1 — Prerequisites

This demo needs Docker (or a Docker-compatible engine, e.g. Rancher Desktop) and [`dive`](https://github.com/wagoodman/dive), a CLI for exploring image layers, used in Step 6.

Install `dive` with Homebrew if the check below reports it missing:

```sh
brew install dive
```

In [ ]:
%%bash
echo "=== Docker ==="
docker --version
echo ""
echo "=== dive ==="
dive --version 2>/dev/null || echo "dive not found on PATH — install it before Step 6"


---

## Step 2 — Build the "Before" — Single-Stage Image

`Dockerfile-single-stage` builds everything — including the full Go toolchain — into the final image, based on `golang:1.22`.

| File | Builds |
|---|---|
| `Dockerfile-single-stage` | `hello-go:single` |

In [ ]:
%%bash
docker build -f Dockerfile-single-stage -t hello-go:single .


---

## Step 3 — Build the "After" — Multi-Stage Image

`Dockerfile-multistage` compiles the binary in a `builder` stage (`golang:1.22`) and copies only the compiled binary into a slim `alpine:3.19` runtime stage. `CGO_ENABLED=0` keeps the binary statically linked so it runs on Alpine's musl libc instead of failing with a missing glibc.

| File | Builds |
|---|---|
| `Dockerfile-multistage` | `hello-go:multi` |

In [ ]:
%%bash
docker build -f Dockerfile-multistage -t hello-go:multi .


---

## Step 4 — Compare Image Sizes

List both images side by side to see how much smaller the multi-stage build is.

In [ ]:
%%bash
docker images | grep hello-go


---

## Step 5 — Tag the Image

Both images share the `hello-go` repository name but have different tags (`single` vs `multi`). Add a `latest` tag pointing at the multi-stage image.

In [ ]:
%%bash
docker tag hello-go:multi hello-go:latest


In [ ]:
%%bash
docker images | grep hello-go


---

## Step 6 — Inspect Image Layers with Dive

`dive` opens an interactive terminal UI showing each image's layers and their size contribution — useful for seeing exactly where the single-stage image's bloat comes from compared to the multi-stage image.

> **Note:** `dive` is a full-screen terminal UI. It won't render inside a notebook output cell — run these commands in a real terminal instead, then return here to continue.

In [ ]:
%%bash
dive hello-go:single


In [ ]:
%%bash
dive hello-go:multi


---

## Step 7 — Run and Verify the App

Run the multi-stage image and confirm the HTTP server responds. `--rm` cleans up the container automatically once it's stopped, and `-p 8080:8080` forwards the container's port to `localhost`.

> **Notebook adaptation:** the source instructions run `docker run` in the foreground and `curl` in a second terminal. Since a notebook cell can't run two things at once, the container below is started detached (`-d`) with a fixed `--name` so the next cells can reach and later stop it.

In [ ]:
%%bash
docker run --rm -d -p 8080:8080 --name hello-go-demo hello-go:multi
sleep 2
docker ps --filter name=hello-go-demo


In [ ]:
%%bash
curl localhost:8080


---

## Teardown

Stop the demo container. Because it was started with `--rm`, stopping it also removes it.

In [ ]:
%%bash
docker stop hello-go-demo


Remove the images built in Steps 2, 3, and 5 (`hello-go:single`, `hello-go:multi`, `hello-go:latest`).

In [ ]:
%%bash
docker rmi hello-go:single hello-go:multi hello-go:latest
